In [67]:
# !pip install gensim nltk scikit-learn matplotlib wordcloud

import re
import logging
import warnings
import collections
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from wordcloud import WordCloud

import nltk
for pkg in ('punkt', 'punkt_tab', 'stopwords'):
    nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

warnings.filterwarnings('ignore')
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                    level=logging.INFO)

OUT_DIR = Path('visualisations')
OUT_DIR.mkdir(exist_ok=True)

print('All imports OK')
import gensim
print(f'Gensim version : {gensim.__version__}')

All imports OK
Gensim version : 4.4.0


### Configuration

In [68]:
CORPUS_PATH = '/Users/jahanvigajera/Desktop/Assignment/NLU_assignment/Assignment2/problem1/dataset_cleaned_final.txt'

# ── Hyperparameter grid ───────────────────────────────────
# (config_name, vector_size, window, negative)
CONFIGS = [
    ('Config-A (dim=50,  win=3, neg=5)',   50,  3,  5),
    ('Config-B (dim=100, win=5, neg=5)',  100,  5,  5),
    ('Config-C (dim=300, win=7, neg=10)', 300,  7, 7),
]

EPOCHS        = 30
MIN_COUNT     = 1      
WORKERS       = 4      # parallel threads
SEED          = 42

# words to analyse
QUERY_WORDS = ['research', 'student', 'phd', 'exam', 'university']

# Analogy experiments: a - b + c = ?
ANALOGIES = [
    ('pg',        'ug',       'btech',   'UG:BTech :: PG:?'),
    ('professor', 'student',  'exam',    'student:exam :: professor:?'),
    ('mtech',     'phd',      'thesis',  'PhD:thesis :: MTech:?'),
    ('student',   'research', 'lab',     'research:lab :: student:?'),
]

print('Configuration ready.')
print(f'  Corpus  : {CORPUS_PATH}')
print(f'  Configs : {len(CONFIGS)} × 2 architectures = {len(CONFIGS)*2} models')

Configuration ready.
  Corpus  : /Users/jahanvigajera/Desktop/Assignment/NLU_assignment/Assignment2/problem1/dataset_cleaned_final.txt
  Configs : 3 × 2 architectures = 6 models


### Corpus Loading & Cleaning

In [69]:
# Domain words to always keep (even if in stoplist)
DOMAIN_KEEP = {
    'exam', 'research', 'student', 'phd', 'university',
    'professor', 'thesis', 'degree', 'course', 'lecture',
    'paper', 'journal', 'conference', 'lab', 'study',
    'ug', 'pg', 'btech', 'mtech', 'undergraduate',
    'postgraduate', 'program', 'faculty', 'department',
    'semester', 'project', 'grade',
}
STOP = set(stopwords.words('english')) - DOMAIN_KEEP


def is_garbage_word(word):
    """Detect random noise tokens like uwkupuusshessen, HmHHHoMogUe."""
    if re.search(r'(.)\1{3,}', word):          # 4+ repeated chars: ssss, llll
        return True
    if len(word) > 6:                           # random caps: HmHHHoMogUe
        upper = sum(1 for c in word if c.isupper())
        if upper / len(word) > 0.4:
            return True
    vowels = set('aeiouAEIOU')
    if len(word) > 5 and not any(c in vowels for c in word):  # no vowels
        return True
    if re.search(r'[bcdfghjklmnpqrstvwxyz]{5,}', word.lower()):  # 5+ consonants
        return True
    return False


def clean_and_tokenise(text):
    """Full cleaning pipeline → list of token lists."""
    corpus = []
    for sent in sent_tokenize(text):
        sent = sent.lower()
        sent = re.sub(r'http\S+|www\.\S+', ' ', sent)   # remove URLs
        sent = re.sub(r'\S+@\S+\.\S+',     ' ', sent)   # remove emails
        sent = re.sub(r'[^a-z\s]',          ' ', sent)   # keep only letters
        sent = re.sub(r'\s+',               ' ', sent).strip()

        tokens = [
            t for t in word_tokenize(sent)
            if t not in STOP
            and len(t) >= 2
            and not is_garbage_word(t)
        ]
        if len(tokens) >= 5:
            corpus.append(tokens)
    return corpus


# Load corpus
p = Path(CORPUS_PATH)
if not p.exists():
    raise FileNotFoundError(f'{CORPUS_PATH} not found. Upload your corpus file.')

raw_text = p.read_text(encoding='utf-8', errors='ignore')
print(f'Raw file size  : {p.stat().st_size/1e6:.1f} MB')

corpus     = clean_and_tokenise(raw_text)
all_tokens = [t for s in corpus for t in s]

# Save cleaned corpus
Path('cleaned_corpus_final.txt').write_text(
    '\n'.join(' '.join(s) for s in corpus), encoding='utf-8'
)

print(f'\nTask 1 — Corpus Stats:')
print(f'  Sentences    : {len(corpus):,}')
print(f'  Total tokens : {len(all_tokens):,}')
print(f'  Unique tokens: {len(set(all_tokens)):,}')
print(f'\nSample: {corpus[0][:15]}')

Raw file size  : 0.4 MB

Task 1 — Corpus Stats:
  Sentences    : 1
  Total tokens : 53,763
  Unique tokens: 7,426

Sample: ['student', 'wellbeing', 'committee', 'iit', 'jodhpur', 'embark', 'journey', 'aasmaan', 'limitless', 'sky', 'wellbeing', 'newsletter', 'previous', 'next', 'mental']


### Corpus Statistics & WordCloud

In [70]:
# Word frequency distribution
freq = collections.Counter(all_tokens)

# Top 20 words bar chart
top20      = freq.most_common(20)
top_words  = [w for w,_ in top20]
top_counts = [c for _,c in top20]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
axes[0].barh(top_words[::-1], top_counts[::-1], color='#2A9D8F')
axes[0].set_title('Top 20 Most Frequent Words', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Frequency')
axes[0].set_facecolor('#F8F9FA')
axes[0].grid(True, axis='x', alpha=0.4)

# WordCloud
wc = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='viridis',
    max_words=150,
).generate_from_frequencies(freq)
axes[1].imshow(wc, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Word Cloud — Corpus Vocabulary', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_DIR/'corpus_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUT_DIR}/corpus_stats.png')

# Check important query words exist
print(f'\nQuery word frequencies:')
for w in QUERY_WORDS:
    count = freq.get(w, 0)
    status = "yes" if count >= MIN_COUNT else ' RARE'
    print(f'  {w:<15} {count:>6} occurrences  {status}')

Saved → visualisations/corpus_stats.png

Query word frequencies:
  research           370 occurrences  yes
  student             80 occurrences  yes
  phd                 38 occurrences  yes
  exam                 3 occurrences  yes
  university          77 occurrences  yes


### Train All 6 Models (Gensim Word2Vec)


In [71]:
import time

trained_models = {}   # key: run_name → gensim Word2Vec model
results_table  = []
Path('models').mkdir(exist_ok=True)
for cfg_name, dim, win, neg in CONFIGS:
    for arch, sg_flag in [('CBOW', 0), ('Skip-gram', 1)]:
        run_name = f'{arch} | {cfg_name}'
        print(f'\nTraining: {run_name}')

        t0 = time.time()

        model = Word2Vec(
            sentences   = corpus,
            vector_size = dim,       # embedding dimension
            window      = win,       # context window size
            negative    = neg,       # number of negative samples
            sg          = sg_flag,   # 0=CBOW, 1=Skip-gram
            hs          = 0,         # 0=negative sampling, 1=hierarchical softmax
            min_count   = 1, # ignore rare words
            workers     = WORKERS,   # parallel threads
            epochs      = EPOCHS,
            seed        = SEED,
            compute_loss=True
        )

        elapsed = time.time() - t0
        vocab_size = len(model.wv)

        loss = model.get_latest_training_loss()

        trained_models[run_name] = model
        results_table.append(dict(
            Architecture = arch,
            Config       = cfg_name,
            Dim = dim, Window = win, Negative = neg,
            VocabSize    = vocab_size,
            TimeSec      = round(elapsed, 2),
            Loss         = round(loss, 2)/len(all_tokens)
        ))

        # Save model
        safe = run_name.replace(' ','_').replace('|','').replace('(','').replace(')','').replace(',','')
        model.save(f'models/{safe}.model')
        print(f'   Done in {elapsed:.1f}s | vocab={vocab_size:,} | loss={loss:.2f}')


# Summary table
print('\n' + '='*75)
print('TASK 2 — TRAINING SUMMARY')
print('='*75)
print(f'{"Arch":<12} {"Config":<40} {"Dim":>5} {"Win":>4} {"Neg":>4} {"Vocab":>8} {"Time":>7} {"Loss":>6}')
print('-'*75)
for r in results_table:
    print(f'{r["Architecture"]:<12} {r["Config"]:<40} '
          f'{r["Dim"]:>5} {r["Window"]:>4} {r["Negative"]:>4} '
          f'{r["VocabSize"]:>8,} {r["TimeSec"]:>6,} {r["Loss"]:>6}s')

2026-03-25 20:03:49,724 : INFO : collecting all words and their counts
2026-03-25 20:03:49,725 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-03-25 20:03:49,736 : INFO : collected 7426 word types from a corpus of 53763 raw words and 1 sentences
2026-03-25 20:03:49,737 : INFO : Creating a fresh vocabulary
2026-03-25 20:03:49,745 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 7426 unique words (100.00% of original 7426, drops 0)', 'datetime': '2026-03-25T20:03:49.745853', 'gensim': '4.4.0', 'python': '3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.0.13.3)]', 'platform': 'macOS-26.3.1-arm64-arm-64bit', 'event': 'prepare_vocab'}
2026-03-25 20:03:49,746 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 53763 word corpus (100.00% of original 53763, drops 0)', 'datetime': '2026-03-25T20:03:49.746466', 'gensim': '4.4.0', 'python': '3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.0.


Training: CBOW | Config-A (dim=50,  win=3, neg=5)


2026-03-25 20:03:49,919 : INFO : EPOCH 14: training on 53763 raw words (10000 effective words) took 0.0s, 1139141 effective words/s
2026-03-25 20:03:49,926 : INFO : EPOCH 15: training on 53763 raw words (10000 effective words) took 0.0s, 1699368 effective words/s
2026-03-25 20:03:49,935 : INFO : EPOCH 16: training on 53763 raw words (10000 effective words) took 0.0s, 1317878 effective words/s
2026-03-25 20:03:49,942 : INFO : EPOCH 17: training on 53763 raw words (10000 effective words) took 0.0s, 1676001 effective words/s
2026-03-25 20:03:49,951 : INFO : EPOCH 18: training on 53763 raw words (10000 effective words) took 0.0s, 1494666 effective words/s
2026-03-25 20:03:49,958 : INFO : EPOCH 19: training on 53763 raw words (10000 effective words) took 0.0s, 1685358 effective words/s
2026-03-25 20:03:49,965 : INFO : EPOCH 20: training on 53763 raw words (10000 effective words) took 0.0s, 1553539 effective words/s
2026-03-25 20:03:49,972 : INFO : EPOCH 21: training on 53763 raw words (1000

   Done in 0.3s | vocab=7,426 | loss=937642.88

Training: Skip-gram | Config-A (dim=50,  win=3, neg=5)


2026-03-25 20:03:50,253 : INFO : EPOCH 9: training on 53763 raw words (10000 effective words) took 0.0s, 647560 effective words/s
2026-03-25 20:03:50,269 : INFO : EPOCH 10: training on 53763 raw words (10000 effective words) took 0.0s, 687632 effective words/s
2026-03-25 20:03:50,286 : INFO : EPOCH 11: training on 53763 raw words (10000 effective words) took 0.0s, 635544 effective words/s
2026-03-25 20:03:50,303 : INFO : EPOCH 12: training on 53763 raw words (10000 effective words) took 0.0s, 611059 effective words/s
2026-03-25 20:03:50,320 : INFO : EPOCH 13: training on 53763 raw words (10000 effective words) took 0.0s, 610373 effective words/s
2026-03-25 20:03:50,337 : INFO : EPOCH 14: training on 53763 raw words (10000 effective words) took 0.0s, 627999 effective words/s
2026-03-25 20:03:50,354 : INFO : EPOCH 15: training on 53763 raw words (10000 effective words) took 0.0s, 652209 effective words/s
2026-03-25 20:03:50,370 : INFO : EPOCH 16: training on 53763 raw words (10000 effect

...

   Done in 1.1s | vocab=7,426 | loss=3861984.50

Training: CBOW | Config-C (dim=300, win=7, neg=10)


2026-03-25 20:03:52,316 : INFO : EPOCH 7: training on 53763 raw words (10000 effective words) took 0.0s, 509923 effective words/s
2026-03-25 20:03:52,341 : INFO : EPOCH 8: training on 53763 raw words (10000 effective words) took 0.0s, 421948 effective words/s
2026-03-25 20:03:52,362 : INFO : EPOCH 9: training on 53763 raw words (10000 effective words) took 0.0s, 493086 effective words/s
2026-03-25 20:03:52,382 : INFO : EPOCH 10: training on 53763 raw words (10000 effective words) took 0.0s, 525061 effective words/s
2026-03-25 20:03:52,402 : INFO : EPOCH 11: training on 53763 raw words (10000 effective words) took 0.0s, 557160 effective words/s
2026-03-25 20:03:52,422 : INFO : EPOCH 12: training on 53763 raw words (10000 effective words) took 0.0s, 532545 effective words/s
2026-03-25 20:03:52,442 : INFO : EPOCH 13: training on 53763 raw words (10000 effective words) took 0.0s, 573810 effective words/s
2026-03-25 20:03:52,462 : INFO : EPOCH 14: training on 53763 raw words (10000 effectiv

   Done in 0.7s | vocab=7,426 | loss=950489.06

Training: Skip-gram | Config-C (dim=300, win=7, neg=10)


2026-03-25 20:03:53,057 : INFO : EPOCH 1: training on 53763 raw words (10000 effective words) took 0.1s, 98598 effective words/s
2026-03-25 20:03:53,157 : INFO : EPOCH 2: training on 53763 raw words (10000 effective words) took 0.1s, 100741 effective words/s
2026-03-25 20:03:53,255 : INFO : EPOCH 3: training on 53763 raw words (10000 effective words) took 0.1s, 103499 effective words/s
2026-03-25 20:03:53,354 : INFO : EPOCH 4: training on 53763 raw words (10000 effective words) took 0.1s, 101707 effective words/s
2026-03-25 20:03:53,450 : INFO : EPOCH 5: training on 53763 raw words (10000 effective words) took 0.1s, 106124 effective words/s
2026-03-25 20:03:53,543 : INFO : EPOCH 6: training on 53763 raw words (10000 effective words) took 0.1s, 109131 effective words/s
2026-03-25 20:03:53,636 : INFO : EPOCH 7: training on 53763 raw words (10000 effective words) took 0.1s, 108866 effective words/s
2026-03-25 20:03:53,730 : INFO : EPOCH 8: training on 53763 raw words (10000 effective word

   Done in 2.9s | vocab=7,426 | loss=4999541.50

TASK 2 — TRAINING SUMMARY
Arch         Config                                     Dim  Win  Neg    Vocab    Time   Loss
---------------------------------------------------------------------------
CBOW         Config-A (dim=50,  win=3, neg=5)            50    3    5    7,426   0.31 17.440300578464743s
Skip-gram    Config-A (dim=50,  win=3, neg=5)            50    3    5    7,426   0.57 55.30924613581831s
CBOW         Config-B (dim=100, win=5, neg=5)           100    5    5    7,426   0.35 16.439287986161485s
Skip-gram    Config-B (dim=100, win=5, neg=5)           100    5    5    7,426   1.12 71.83350073470602s
CBOW         Config-C (dim=300, win=7, neg=10)          300    7    7    7,426   0.68 17.679241485780185s
Skip-gram    Config-C (dim=300, win=7, neg=10)          300    7    7    7,426   2.86 92.99223443632238s


### Training Time Comparison Chart

In [72]:
labels  = [f"{r['Architecture']}\n{r['Config'].split('(')[1].rstrip(')')}" for r in results_table]
times   = [r['TimeSec'] for r in results_table]
colors  = ['#E63946' if r['Architecture']=='CBOW' else '#2A9D8F' for r in results_table]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(labels)), times, color=colors, alpha=0.85, edgecolor='white')

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{t:.1f}s', ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Training Time (seconds)')
ax.set_title('Training Time: All Configs & Architectures', fontsize=13, fontweight='bold')
ax.set_facecolor('#F8F9FA')
ax.grid(True, axis='y', alpha=0.4)
ax.legend(handles=[
    mpatches.Patch(color='#E63946', label='CBOW'),
    mpatches.Patch(color='#2A9D8F', label='Skip-gram')
], fontsize=10)

plt.tight_layout()
plt.savefig(OUT_DIR/'training_time.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUT_DIR}/training_time.png')

Saved → visualisations/training_time.png


### Top-5 Nearest Neighbours
Using **Config-B** (dim=100, win=5, neg=5) as baseline.

In [73]:
BEST_CBOW = 'CBOW | Config-B (dim=100, win=5, neg=5)'
BEST_SG   = 'Skip-gram | Config-B (dim=100, win=5, neg=5)'


def print_neighbours(model, model_name):
    vocab_set = set(model.wv.index_to_key)
    print(f'\n{"="*62}')
    print(f'  TOP-5 NEAREST NEIGHBOURS — {model_name}')
    print(f'{"="*62}')
    print(f'  {"Word":<15} {"Rank":<5} {"Neighbour":<20} {"Cosine Sim":>10}')
    print('  ' + '-'*52)

    for word in QUERY_WORDS:
        if word not in vocab_set:
            print(f'  {word:<15} [NOT IN VOCABULARY]')
            continue
        # Gensim most_similar returns [(word, score), ...]
        neighbours = model.wv.most_similar(word, topn=5)
        for rank, (nbr, score) in enumerate(neighbours, 1):
            label = word if rank == 1 else ''
            print(f'  {label:<15} {rank:<5} {nbr:<20} {score:>10.4f}')
        print()


print_neighbours(trained_models[BEST_CBOW], 'CBOW      (Gensim)')
print_neighbours(trained_models[BEST_SG],   'Skip-gram (Gensim)')


  TOP-5 NEAREST NEIGHBOURS — CBOW      (Gensim)
  Word            Rank  Neighbour            Cosine Sim
  ----------------------------------------------------
  research        1     available                0.9977
                  2     home                     0.9977
                  3     platform                 0.9977
                  4     free                     0.9976
                  5     institution              0.9972

  student         1     activities               0.9995
                  2     orientation              0.9995
                  3     swc                      0.9994
                  4     event                    0.9994
                  5     organized                0.9993

  phd             1     vibrant                  0.9991
                  2     facilities               0.9989
                  3     batch                    0.9986
                  4     hostels                  0.9986
                  5     counselors               0.998

### Analogy Experiments

In [74]:
def run_analogies(model, model_name):
    vocab_set = set(model.wv.index_to_key)
    print(f'\n{"="*68}')
    print(f'  ANALOGY EXPERIMENTS — {model_name}')
    print(f'  Formula: positive=[a, c], negative=[b]  →  a - b + c = ?')
    print(f'{"="*68}')
    print(f'  {"Analogy":<35} {"Rank":<5} {"Prediction":<20} {"Score":>8}  OK?')
    print('  ' + '-'*68)

    for pos1, neg1, pos2, desc in ANALOGIES:
        missing = [w for w in (pos1, neg1, pos2) if w not in vocab_set]
        if missing:
            print(f'  {desc:<35} [OOV: {", ".join(missing)}]')
            continue
        try:
            # Gensim analogy: positive=[pos1, pos2], negative=[neg1]
            results = model.wv.most_similar(
                positive=[pos1, pos2],
                negative=[neg1],
                topn=3
            )
            for i, (word, score) in enumerate(results):
                label = desc if i == 0 else ''
                flag  = ('✓ yes' if score > 0.55 else
                         '~ maybe' if score > 0.35 else '✗ weak') if i == 0 else ''
                print(f'  {label:<35} {i+1:<5} {word:<20} {score:>8.4f}  {flag}')
            print()
        except Exception as e:
            print(f'  {desc:<35} ERROR: {e}\n')


run_analogies(trained_models[BEST_CBOW], 'CBOW      (Gensim)')
run_analogies(trained_models[BEST_SG],   'Skip-gram (Gensim)')


  ANALOGY EXPERIMENTS — CBOW      (Gensim)
  Formula: positive=[a, c], negative=[b]  →  a - b + c = ?
  Analogy                             Rank  Prediction              Score  OK?
  --------------------------------------------------------------------
  UG:BTech :: PG:?                    1     program                0.9878  ✓ yes
                                      2     st                     0.9871  
                                      3     quantitative           0.9869  

  student:exam :: professor:?         1     sme                    0.9923  ✓ yes
                                      2     deptt                  0.9918  
                                      3     assistant              0.9913  

  PhD:thesis :: MTech:?               1     centres                0.3579  ~ maybe
                                      2     enabled                0.3534  
                                      3     bioengineering         0.3508  

  research:lab :: student:?           1    

### Cosine Similarity Table

In [75]:
word_pairs = [
    ('research', 'lab'),
    ('student',  'exam'),
    ('phd',      'thesis'),
    ('btech',    'ug'),
    ('professor','faculty'),
]

cbow_m  = trained_models[BEST_CBOW]
sg_m    = trained_models[BEST_SG]
vset_c  = set(cbow_m.wv.index_to_key)
vset_s  = set(sg_m.wv.index_to_key)

print(f'\nCosine Similarity Comparison: CBOW vs Skip-gram (Config-B)')
print(f'{"Word Pair":<28} {"CBOW":>8} {"Skip-gram":>10}  Closer pair')
print('-'*55)
for w1, w2 in word_pairs:
    if w1 not in vset_c or w2 not in vset_c:
        print(f'  ({w1}, {w2})  [OOV]'); continue
    # Gensim .similarity() computes cosine similarity directly
    sc = cbow_m.wv.similarity(w1, w2)
    ss = sg_m.wv.similarity(w1, w2)
    print(f'  ({w1}, {w2})  {"":<{18-len(w1)-len(w2)}} {sc:>8.4f} {ss:>10.4f}  '
          f'{"CBOW" if sc > ss else "Skip-gram"}')


Cosine Similarity Comparison: CBOW vs Skip-gram (Config-B)
Word Pair                        CBOW  Skip-gram  Closer pair
-------------------------------------------------------
  (research, lab)            0.9940     0.6438  CBOW
  (student, exam)            0.9867     0.5772  CBOW
  (phd, thesis)              0.1092    -0.0026  CBOW
  (btech, ug)                0.8976     0.5784  CBOW
  (professor, faculty)       0.9728     0.7211  CBOW


### Cross-Config Nearest Neighbour Comparison

In [76]:
# Compare how 'research' neighbours change across all 6 models
probe_word = 'research'

print(f'\nNearest neighbours for "{probe_word}" across all configs:')
print(f'{"Model":<48} Top-3 Neighbours')
print('-'*80)

for run_name, model in trained_models.items():
    if probe_word not in model.wv:
        print(f'{run_name:<48} [OOV]')
        continue
    nbrs = model.wv.most_similar(probe_word, topn=3)
    nbr_str = ', '.join(f'{w}({s:.3f})' for w, s in nbrs)
    print(f'{run_name:<48} {nbr_str}')


Nearest neighbours for "research" across all configs:
Model                                            Top-3 Neighbours
--------------------------------------------------------------------------------
CBOW | Config-A (dim=50,  win=3, neg=5)          textbooks(0.997), service(0.996), sciences(0.996)
Skip-gram | Config-A (dim=50,  win=3, neg=5)     newspaper(0.943), newspapers(0.932), tools(0.921)
CBOW | Config-B (dim=100, win=5, neg=5)          available(0.998), home(0.998), platform(0.998)
Skip-gram | Config-B (dim=100, win=5, neg=5)     newspapers(0.933), writing(0.912), resources(0.896)
CBOW | Config-C (dim=300, win=7, neg=10)         available(0.999), social(0.999), free(0.999)
Skip-gram | Config-C (dim=300, win=7, neg=10)    newspapers(0.905), newspaper(0.874), unrestricted(0.870)


### PCA Visualisation

In [77]:
# Word clusters for visualisation
CLUSTERS = [
    ('Academic Roles',   '#E63946',
     ['professor','student','phd','researcher','faculty','lecturer','advisor']),
    ('Research',         '#2A9D8F',
     ['research','thesis','paper','journal','conference','methodology','experiment']),
    ('Programme/Degree', '#E9C46A',
     ['ug','pg','btech','mtech','degree','undergraduate','postgraduate','program']),
    ('Assessment',       '#F4A261',
     ['exam','grade','result','evaluation','project','test','assignment']),
    ('University Infra', '#457B9D',
     ['university','department','lab','library','lecture','seminar','campus']),
]


def extract_embeddings(model):
    vocab_set = set(model.wv.index_to_key)
    vecs, words, cols, cats = [], [], [], []
    for label, color, word_list in CLUSTERS:
        for w in word_list:
            if w in vocab_set:
                vecs.append(model.wv[w])   # gensim: model.wv[word]
                words.append(w)
                cols.append(color)
                cats.append(label)
    if not vecs:
        return None, None, None, None
    return normalize(np.array(vecs), norm='l2'), words, cols, cats


def scatter_plot(ax, coords, words, cols, title, method):
    ax.set_facecolor('#F8F9FA')
    ax.grid(True, color='#DEE2E6', linestyle='--', alpha=0.6)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(f'{method}-1', fontsize=9)
    ax.set_ylabel(f'{method}-2', fontsize=9)
    ax.scatter(coords[:,0], coords[:,1], c=cols,
               s=80, alpha=0.85, edgecolors='white', linewidths=0.5, zorder=3)
    for w, (x, y) in zip(words, coords):
        ax.annotate(w, xy=(x,y), xytext=(4,4),
                    textcoords='offset points', fontsize=8)
    handles = [mpatches.Patch(facecolor=c, label=l) for l,c,_ in CLUSTERS]
    ax.legend(handles=handles, loc='upper left', fontsize=7, framealpha=0.85)


# Extract embeddings
V_c, W_c, C_c, cats_c = extract_embeddings(trained_models[BEST_CBOW])
V_s, W_s, C_s, cats_s = extract_embeddings(trained_models[BEST_SG])

# PCA
pca = PCA(n_components=2, random_state=42)
pca_c = pca.fit_transform(V_c)
pca_s = PCA(n_components=2, random_state=42).fit_transform(V_s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
scatter_plot(ax1, pca_c, W_c, C_c, 'PCA — CBOW (Gensim)',      'PCA')
scatter_plot(ax2, pca_s, W_s, C_s, 'PCA — Skip-gram (Gensim)', 'PCA')
plt.suptitle('PCA: CBOW vs Skip-gram', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'pca_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUT_DIR}/pca_comparison.png')

Saved → visualisations/pca_comparison.png


### t-SNE Visualisation

In [78]:
perp   = min(15, len(V_c) - 1)
tsne_c = TSNE(n_components=2, perplexity=perp, init='pca',
              n_iter=1000, random_state=42).fit_transform(V_c)
tsne_s = TSNE(n_components=2, perplexity=perp, init='pca',
              n_iter=1000, random_state=42).fit_transform(V_s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
scatter_plot(ax1, tsne_c, W_c, C_c, 't-SNE — CBOW (Gensim)',      't-SNE')
scatter_plot(ax2, tsne_s, W_s, C_s, 't-SNE — Skip-gram (Gensim)', 't-SNE')
plt.suptitle('t-SNE: CBOW vs Skip-gram', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'tsne_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUT_DIR}/tsne_comparison.png')

Saved → visualisations/tsne_comparison.png


### WordCloud for Each Cluster

In [79]:
# WordCloud showing the most similar words to each cluster's seed word
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

seed_words = ['research', 'student', 'exam', 'university', 'phd']
colors_wc  = ['viridis', 'plasma', 'inferno', 'cividis', 'magma']

for i, (seed, cmap) in enumerate(zip(seed_words, colors_wc)):
    if seed not in sg_m.wv:
        axes[i].set_visible(False)
        continue
    # Get top 50 similar words as frequency dict for wordcloud
    similar = sg_m.wv.most_similar(seed, topn=50)
    wc_freq = {w: score + 1 for w, score in similar}   # shift score to positive
    wc_freq[seed] = 2.0   # seed word largest

    wc = WordCloud(width=500, height=300,
                   background_color='white',
                   colormap=cmap,
                   max_words=50).generate_from_frequencies(wc_freq)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].axis('off')
    axes[i].set_title(f'Similar to: "{seed}"', fontsize=12, fontweight='bold')

axes[-1].set_visible(False)  # hide unused subplot
plt.suptitle('WordClouds — Skip-gram Nearest Neighbours', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUT_DIR}/wordclouds.png')

Saved → visualisations/wordclouds.png


### Cluster Compactness Analysis

In [80]:
def intra_cluster_dist(coords, cats):
    """Mean pairwise Euclidean distance within each cluster."""
    groups = collections.defaultdict(list)
    for coord, cat in zip(coords, cats):
        groups[cat].append(coord)
    return {
        cat: float(np.mean([
            np.linalg.norm(np.array(pts[i]) - np.array(pts[j]))
            for i in range(len(pts)) for j in range(i+1, len(pts))
        ])) if len(pts) > 1 else 0.0
        for cat, pts in groups.items()
    }


for method, cc, cs in [('PCA', pca_c, pca_s), ('t-SNE', tsne_c, tsne_s)]:
    dc = intra_cluster_dist(cc, cats_c)
    ds = intra_cluster_dist(cs, cats_s)
    print(f'\n{method} — Cluster Compactness (↓ = tighter, better clustering)')
    print(f'  {"Cluster":<35} {"CBOW":>8} {"Skip-gram":>10}  Winner')
    print('  ' + '-'*60)
    for cat in dc:
        d1, d2 = dc[cat], ds.get(cat, 0)
        print(f'  {cat:<35} {d1:>8.4f} {d2:>10.4f}  '
              f'{"CBOW" if d1 < d2 else "Skip-gram"}')


PCA — Cluster Compactness (↓ = tighter, better clustering)
  Cluster                                 CBOW  Skip-gram  Winner
  ------------------------------------------------------------
  Academic Roles                        0.6693     0.6752  CBOW
  Research                              0.8059     0.6647  Skip-gram
  Programme/Degree                      0.0247     0.1887  CBOW
  Assessment                            0.7577     0.7401  Skip-gram
  University Infra                      0.0417     0.4740  CBOW

t-SNE — Cluster Compactness (↓ = tighter, better clustering)
  Cluster                                 CBOW  Skip-gram  Winner
  ------------------------------------------------------------
  Academic Roles                        4.0371    16.0939  CBOW
  Research                              2.8903    20.0026  CBOW
  Programme/Degree                      3.1695     9.3545  CBOW
  Assessment                            3.6559    17.1151  CBOW
  University Infra                